## Model with Pytorch

In [28]:
import numpy as np
import pandas as pd
import torch

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)

torch.manual_seed(42)
np.random.seed(42)

## Load Dataset
housing=fetch_california_housing()

X=housing.data
y=housing.target

print("X shape:", X.shape)
print('y shape', y.shape)


X shape: (20640, 8)
y shape (20640,)


In [29]:

# 3. Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# 4. Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 5. Convert To Tensors
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32

).reshape(-1, 1)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

In [30]:
# Difine Architecture
layer_size=[8, 128, 64, 32, 1]

weights=[]
biases=[]

for i in range(len(layer_size)-1):
    W=(
        torch.randn(
            layer_size[i],
            layer_size[i+1]
        )*np.sqrt(2/layer_size[i])
    )
    
    W.requires_grad_()
    
    b=torch.zeros(
        layer_size[i+1],
        requires_grad=True
    )
    
    weights.append(W)
    biases.append(b)
    
    

In [31]:
## Forward function
def model(X):
    a=X
    for W, b in zip(weights[:-1], biases[:-1]):
        z=a @ W+b
        a=torch.relu(z)
        
    output=a@ weights[-1]+biases[-1]
    
    return output

In [32]:
## Training
epochs=1000
learning_rate=0.001

loss_history=[]

for epoch in range(epochs):
    # forward pass
    y_pred=model(X_train)
    
    #MSE loss
    loss=((y_pred-y_train)**2).mean()
    
    # Backward Pass
    loss.backward()
    
    # Gradient Descent
    with torch.no_grad():
        for W in weights:
            W-=learning_rate*W.grad
            
        for b in biases:
            b-=learning_rate*b.grad
            
    # Clear Gradients
    for W in weights:
        W.grad.zero_()
    
    for b in biases:
        b.grad.zero_()
    
    loss_history.append(loss.item())
    
    if epoch % 100==0:
        print(
            f"Epoch {epoch:4d} |"
            f"Loss: {loss.item():.6f}"
        )

Epoch    0 |Loss: 25.692818
Epoch  100 |Loss: 0.963072
Epoch  200 |Loss: 0.845735
Epoch  300 |Loss: 0.778859
Epoch  400 |Loss: 0.732012
Epoch  500 |Loss: 0.696917
Epoch  600 |Loss: 0.669017
Epoch  700 |Loss: 0.645941
Epoch  800 |Loss: 0.626357
Epoch  900 |Loss: 0.609369


In [33]:
# Evaluation
with torch.no_grad():
    test_predictions=model(X_test)
    
predictions=test_predictions.numpy()

mae=mean_absolute_error(
    y_test.numpy(),predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test.numpy(),
        predictions
    )

)

r2 = r2_score(
    y_test.numpy(),
    predictions
)

print("\nResults")
print("-" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


Results
----------------------------------------
MAE  : 0.5645
RMSE : 0.7847
R²   : 0.5301


In [34]:
## Single Prediction Example

sample=X_test[0].reshape(1, -1)

with torch.no_grad():
    prediction=model(sample)
    
    
    
print("\nPredicted House Price:")
print(prediction.item())

print("\nActual House Price:")
print(y_test[0].item())


Predicted House Price:
0.6297639608383179

Actual House Price:
0.47699999809265137


## Sequential API

In [35]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

seq_model=Sequential([
    
    Dense(128, activation="relu",
          input_shape=(8,)),
    
    Dense(64, activation="relu"),
    
    Dense(32, activation="relu"),
    
    Dense(1)
])

seq_model.compile(
    optimizer="adam",
    loss='mse',
    metrics=["mse"]
)

seq_model.summary()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,521 (45.00 KB)

 Trainable params: 11,521 (45.00 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
## Train
history_seq=seq_model.fit(
    X_train, y_train, validation_split=0.2,
    epochs=100,
    batch_size=64,
    verbose=1
)

Epoch 1/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.9728 - mse: 0.9728 - val_loss: 0.4951 - val_mse: 0.4951
Epoch 2/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 740us/step - loss: 0.4021 - mse: 0.4021 - val_loss: 0.4061 - val_mse: 0.4061
Epoch 3/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 635us/step - loss: 0.3622 - mse: 0.3622 - val_loss: 0.3813 - val_mse: 0.3813
Epoch 4/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 617us/step - loss: 0.3520 - mse: 0.3520 - val_loss: 0.3858 - val_mse: 0.3858
Epoch 5/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 606us/step - loss: 0.3412 - mse: 0.3412 - val_loss: 0.3578 - val_mse: 0.3578
Epoch 6/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3237 - mse: 0.3237 - val_loss: 0.3510 - val_mse: 0.3510
Epoch 7/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 732us/step - loss: 0.3136 - mse: 0.3136 - val_loss: 0.3423 - val_mse: 0.3423
Epoch 8/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 633us/step - loss: 0.3086 - mse: 0.3086 - val_loss: 0.3440 - val_mse: 0.3440
Epoch 9/100
207/207 ━━━━━━━━━━━━━━━━

## Functional API

In [37]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense

inputs=Input(
    shape=(8,),
    name="input_layer"
)

x=Dense(128, activation="relu")(inputs)

x=Dense(64, activation="relu")(x)

x=Dense(32, activation="relu")(x)

outputs=Dense(1, name="output_layer")(x)

func_model=Model(
    inputs=inputs,
    outputs=outputs,
    name="housing_functional"
)

func_model.compile(
    optimizer='adam',
    loss="mse",
    metrics=["mae"]
)

func_model.summary()

Model: "housing_functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,521 (45.00 KB)

 Trainable params: 11,521 (45.00 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
## Train Functional API

history_func=func_model.fit(
    X_train, y_train, validation_split=0.2,
    epochs=100,
    batch_size=64,
    verbose=1
)

Epoch 1/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.1725 - mae: 0.7284 - val_loss: 0.4815 - val_mae: 0.4946
Epoch 2/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step - loss: 0.4061 - mae: 0.4522 - val_loss: 0.4145 - val_mae: 0.4438
Epoch 3/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 639us/step - loss: 0.3625 - mae: 0.4261 - val_loss: 0.3828 - val_mae: 0.4264
Epoch 4/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 632us/step - loss: 0.3432 - mae: 0.4135 - val_loss: 0.3762 - val_mae: 0.4393
Epoch 5/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 614us/step - loss: 0.3308 - mae: 0.4042 - val_loss: 0.3741 - val_mae: 0.4358
Epoch 6/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 649us/step - loss: 0.3243 - mae: 0.3993 - val_loss: 0.3481 - val_mae: 0.4116
Epoch 7/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 884us/step - loss: 0.3150 - mae: 0.3927 - val_loss: 0.3371 - val_mae: 0.3951
Epoch 8/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 655us/step - loss: 0.3241 - mae: 0.3915 - val_loss: 0.3442 - val_mae: 0.4054
Epoch 9/100
207/207 ━━━━━━━━━━━━━━

## Subclassing API

In [39]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense

class HousingModel(Model):
    def __init__(self):
        super().__init__()
        self.hidden1=Dense(
            128, activation="relu"
        )
        
        self.hidden2=Dense(
            64, activation="relu"
        )
        
        self.hidden3=Dense(
            32, activation="relu"
        )
        
        self.output_layer=Dense(1)
        
    def call(self, inputs):
        x=self.hidden1(inputs)
        
        x=self.hidden2(x)
        
        x=self.hidden3(x)
        
        return self.output_layer(x)


In [40]:
## Subclass model

sub_model=HousingModel()
sub_model.build(
    input_shape=(None, 8)
)
sub_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

sub_model.summary()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:427: UserWarning: `build()` was called on layer 'housing_model_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "housing_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_22 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [41]:
## Train
history_sub=sub_model.fit(
    X_train, y_train, 
    validation_split=0.2,
    epochs=100,
    batch_size=64, 
    verbose=1
)

Epoch 1/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 1.2932 - mae: 0.6668 - val_loss: 0.4724 - val_mae: 0.4909
Epoch 2/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - loss: 0.4134 - mae: 0.4512 - val_loss: 0.4110 - val_mae: 0.4453
Epoch 3/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 618us/step - loss: 0.3627 - mae: 0.4278 - val_loss: 0.3806 - val_mae: 0.4392
Epoch 4/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 694us/step - loss: 0.3462 - mae: 0.4174 - val_loss: 0.3700 - val_mae: 0.4359
Epoch 5/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - loss: 0.3331 - mae: 0.4071 - val_loss: 0.3595 - val_mae: 0.4178
Epoch 6/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - loss: 0.3227 - mae: 0.3996 - val_loss: 0.3598 - val_mae: 0.4345
Epoch 7/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 613us/step - loss: 0.3173 - mae: 0.3939 - val_loss: 0.3414 - val_mae: 0.4128
Epoch 8/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 703us/step - loss: 0.3084 - mae: 0.3887 - val_loss: 0.3443 - val_mae: 0.4110
Epoch 9/100
207/207 ━━━━━━━━━━━━━━

## Evaluation

In [42]:
def evaluate_model(model, X_test, y_test):

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    return mae, rmse, r2

In [43]:
models = {
    "Sequential": seq_model,
    "Functional": func_model,
    "Subclassing": sub_model
}

for name, model in models.items():

    mae, rmse, r2 = evaluate_model(
        model,
        X_test,
        y_test
    )

    print(f"\n{name}")
    print("-" * 30)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 276us/step

Sequential
------------------------------
MAE  : 0.3525
RMSE : 0.5225
R²   : 0.7917
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 286us/step

Functional
------------------------------
MAE  : 0.3530
RMSE : 0.5430
R²   : 0.7750
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 250us/step

Subclassing
------------------------------
MAE  : 0.3458
RMSE : 0.5187
R²   : 0.7947
